In [30]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [31]:
import pandas as pd

role_profiles = pd.read_csv("skill_gap/role_skill_profiles.csv")


In [32]:
def get_top_role_skills(role_id, top_n=7):
    role_skills = role_profiles[
        role_profiles["role_id"] == role_id
    ].sort_values("importance", ascending=False)

    return set(role_skills.head(top_n)["skill_id"])


In [33]:
def get_progression_delta(current_role, next_role, top_n=7):
    current_skills = get_top_role_skills(current_role, top_n)
    next_skills = get_top_role_skills(next_role, top_n)

    return next_skills - current_skills


In [34]:
def calculate_progression_readiness(user_skills, current_role, next_role, top_n=7):
    delta_skills = get_progression_delta(current_role, next_role, top_n)

    if not delta_skills:
        return 1.0, []

    matched = set(user_skills) & delta_skills
    readiness = round(len(matched) / len(delta_skills), 2)

    missing = list(delta_skills - matched)
    return readiness, missing


In [35]:
def simulate_career_path(domain, current_role, user_skill_ids, top_n=7):
    ladder = CAREER_LADDERS.get(domain)
    if not ladder:
        return {"error": "Unknown career domain"}

    if current_role not in ladder:
        return {"error": "Role not in ladder"}

    idx = ladder.index(current_role)
    if idx + 1 >= len(ladder):
        return {"message": "Already at highest role"}

    next_role = ladder[idx + 1]

    readiness, missing_skills = calculate_progression_readiness(
        user_skills=user_skill_ids,
        current_role=current_role,
        next_role=next_role,
        top_n=top_n
    )

    return {
        "current_role": current_role,
        "next_role": next_role,
        "readiness_score": readiness,
        "missing_progression_skills": missing_skills
    }


In [36]:
user_skills = {"SK007", "SK102", "SK041"}

simulate_career_path(
    domain="SOFTWARE_ENGINEERING",
    current_role="JR_SE",
    user_skill_ids=user_skills,
    top_n=7
)


{'current_role': 'JR_SE',
 'next_role': 'SE',
 'readiness_score': 1.0,
 'missing_progression_skills': []}